# Cross section extraction

Used in parallel with `xsec_analyzer` (additional uncertainties) to extract and plot (with systematics) the nuNCQE cross section. 

While you can extract the event rates from `xsec_analyzer`, since it's a 1D measurement there is no unfolding required, and I instead can grab it directly from the `05_FULL_DATA_signal_band.ipynb` script (one of the cells outputs the MC predictions, observed events, estimated skyshine, and off-beam events). 

In [ ]:
'''
Running that script over the base and Empirical model GENIE files yields:

# Base GENIE
==============================
POT SCALED
==============================
MC POT: 4.426
Total NCQE true signal: 5871.5092634432895
--------------------
NCQE (signal) 204.63
NCother 45.93
CC 12.19
nubarNCQE 5.23
OOFV 287.07
External 81.86
Off-beam: 127.43794062670655
Skyshine: 14.1
--------------------
Total Predicted: 778.4380359772886
Total Observed: 772


# 016 GENIE
==============================
POT SCALED
==============================
MC POT: 4.41733
Total NCQE true signal: 5883.033416113353
--------------------
NCQE (signal) 152.61
NCother 47.76
CC 7.0
nubarNCQE 2.91
OOFV 253.39
External 78.23
Off-beam: 127.43794062670655
Skyshine: 14.1
--------------------
Total Predicted: 683.4356317785227
Total Observed: 772
'''

In [1]:
from lib.imports import *

# ==========================================
# INPUTS
# ==========================================

# Takes inputs from the FULL_DATA_signal_band.ipynb script
# Counts should be POT (for MC), trigger (for off-beam), and exposure (for skyshine) scaled
# MC-predicted counts should be WEIGHTED with the CV's using the MicroBooNE v3.0.6 tune

# Data parameters
N_DATA_RECO = 772         # observed reconstructed events

# Off-beam + Skyshine parameters
N_OFF_BEAM = 127.44       # scaled off-beam in full data
N_SKYSHINE = (14.1)       # calculated skyshine in signal band

# MC parameters
SIGMA_GENIE = 2.04334       # obtained from the GENIE flattree
MC_TRUE_SIGNAL = 5883       # true signal rate (POT-scaled)

# reco:
MC_NCQE = 152.61
MC_NCother = 47.76
MC_CC = 7.0
MC_nubar = 2.91
MC_OOFV = 253.39
MC_external = 78.23         # dirt events scaled by alpha

# predicted total MC reco events from GENIE
MC_TOTAL_RECO = MC_NCQE + MC_NCother + MC_CC + MC_nubar + MC_OOFV + MC_external
MC_TOTAL_BKG = MC_NCother + MC_CC + MC_nubar + MC_OOFV + MC_external

MC_EFFICIENCY = MC_NCQE / MC_TRUE_SIGNAL  # calculated efficiency (NCQE reco, true --> NCQE true)

# ==========================================
# CALCULATIONS
# ==========================================

total_bkg = MC_TOTAL_BKG + N_OFF_BEAM + N_SKYSHINE

# 4. Cross Section & Scale Factor
scale_factor = (N_DATA_RECO - total_bkg) / MC_NCQE
sigma_obs = SIGMA_GENIE * scale_factor

# ==========================================
# OUTPUT
# ==========================================

print("--- EVENT BREAKDOWN ---")
print(f"Reconstructed events (Data):       {N_DATA_RECO}")
print(f"Total Expected Background:         {total_bkg:.2f}")
print("----------------------------------")
print(f"Off-beam background:               {N_OFF_BEAM:.2f}")
print(f"Fitted skyshine background:        {N_SKYSHINE:.2f}")
print(f"Predicted MC background:           {MC_TOTAL_BKG:.2f}")
print("----------------------------------")
print(f"Total Predicted MC Signal:         {MC_TRUE_SIGNAL:.2f}")
print(f"Predicted Reco MC Signal:          {MC_NCQE:.2f}")
print(f"Efficiency:                        {100* (MC_NCQE / MC_TRUE_SIGNAL):.2f}%")
print("----------------------------------")
print("--- RESULTS ---")
print(f"Observed XS:                       {sigma_obs:.5f}")
print(f"Scale factor (Data/GENIE):         {scale_factor:.3f}")

--- EVENT BREAKDOWN ---
Reconstructed events (Data):       772
Total Expected Background:         530.83
----------------------------------
Off-beam background:               127.44
Fitted skyshine background:        14.10
Predicted MC background:           389.29
----------------------------------
Total Predicted MC Signal:         5883.00
Predicted Reco MC Signal:          152.61
Efficiency:                        2.59%
----------------------------------
--- RESULTS ---
Observed XS:                       3.22910
Scale factor (Data/GENIE):         1.580


In [ ]:
# Investigating systematic impact on the cross section (adjust above script and re-calculate XS)

def XS_systematic(nominal, adjusted):
    print(f"{round(100*np.abs(adjusted - nominal) / nominal,2)}%")

nominal_XS = 3.229        # CV extraction
adjusted_XS = 3.01379     # altered/systematic investigation

XS_systematic(nominal_XS, adjusted_XS)

In [6]:
# Plotting

def plot_official_xs(CV_val, CV_label, data_val, BNB_stat_err,
                     total_err_up, total_err_down, 
                     pot_string, label_prefix, plot_name, stat_only,
                     deexc_alt_val=None, deexc_alt_label='Base GENIE de-excitation'):
    """
    ... (existing docstring) ...
    deexc_alt_val (float, optional): Result using an alternate de-excitation
        model in the unfolding (e.g. base GENIE instead of the Empirical Model).
        Shown as a band + open marker, separate from stat/sys error bars.
    deexc_alt_label (str): Legend label for the alt de-excitation marker.
    """
    fig, ax = plt.subplots(figsize=(3.2, 5))
    
    if stat_only:
        print('\nONLY STAT (BNB + MC) ERRORS')
        total_label = f'{label_prefix} (stat. + MC stat.)'
    else:
        total_label = f'{label_prefix} (stat. + sys.)'

    stat_err = (BNB_stat_err/100) * data_val
    tot_err_up   = (total_err_up   / 100) * data_val
    tot_err_down = (total_err_down / 100) * data_val

    # --- De-excitation model dependence (draw first -> lowest zorder) ---
    if deexc_alt_val is not None:
        lo, hi = sorted([data_val, deexc_alt_val])
        ax.plot(0.5, deexc_alt_val, marker='D', markerfacecolor='none',
                markeredgecolor='#FF3366', markeredgewidth=1.2, markersize=6,
                linestyle='none', zorder=4, label = 'Base GENIE de-exc.')

    # --- Inner error bar (Stat only, symmetric) ---
    ax.errorbar(0.5, data_val, yerr=stat_err,
            fmt='ko', capsize=5, label=total_label, markersize=5, zorder=5)

    # --- Outer error bar (Total, asymmetric) ---
    ax.errorbar(0.5, data_val, yerr=[[tot_err_down], [tot_err_up]],
            fmt='none', ecolor='k', capsize=5, zorder=3)

    # --- Prediction (GENIE CV) ---
    ax.axhline(CV_val, label=CV_label, color='blue',
               linestyle='dashdot', linewidth=1.5, zorder=2)

    # --- 3. Formatting Ticks & Scale ---
    ax.tick_params(axis='y', which='both', direction='in', right=True, labelsize=14)
    ax.yaxis.set_major_locator(MultipleLocator(0.5e-38))
    ax.yaxis.set_minor_locator(MultipleLocator(0.1e-38))
    ax.set_xticks([]) # Remove x-axis ticks
    ax.set_xlim(0, 1) # Set arbitrary bounds to center the point at 0.5

    # Force scientific notation 10^-38
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0,0))
    ax.yaxis.set_major_formatter(formatter)

    # --- 4. Labels & Legend ---
    plt.ylabel(r'$\nu$' + 'NCQE Cross Section [cm' + r'$^2$' + ' / oxygen]', fontsize=14)
    plt.ylim([0, 5.5e-38])
    
    # Legend - frameon=False as requested
    plt.legend(frameon=False, fontsize=9, loc='upper right')

    # --- 5. Bottom Text (POT & Experiment Info) ---
    # Using weight='bold' to match the MicroBooNE style
    bottom_text = f"ANNIE {label_prefix}\n{pot_string} POT"
    ax.text(
        0.05, 0.05, bottom_text,
        transform=ax.transAxes,
        fontsize=11,
        fontweight='bold',
        verticalalignment='bottom'
    )
    
    
    plt.tight_layout()
    plt.savefig(plot_name, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor = 'w')
    plt.show()
    
    print('Plot saved to: plots/' + plot_name)


# ==========================================
# Asymmetric uncertainty combination
# ==========================================
def quad_sys_asymmetric(up_uncs, down_uncs):
    """Combine asymmetric systematics in quadrature separately"""
    return np.sqrt(np.sum(np.array(up_uncs)**2)), \
           np.sqrt(np.sum(np.array(down_uncs)**2))

# Systematics — specify up/down separately
# Format: (up%, down%) — symmetric systematics get same value both sides
systematics = {
                  # up,   down 
    'flux':        (24.8, 24.8),
    'genie_xsec':  (24.6, 24.6),
    'MC_stat':     (6.7, 6.7),
    'skyshine':    (5.4, 5.4),
    'dirt':        (6.7,  6.7),
    'primary':     (6.4,  21.0),
    'secondary':   (4.6,  4.6),
    'DV':          (8.3,  8.3),
    'POT':         (2.0,  2.0),
}

up_sys   = [v[0] for v in systematics.values()]
down_sys = [v[1] for v in systematics.values()]

sys_up, sys_down = quad_sys_asymmetric(up_sys, down_sys)

STAT_ERROR = 100*np.sqrt(N_DATA_RECO)/N_DATA_RECO
#STAT_ERROR = 0     # enable for syst-only

TOTAL_UP   = np.sqrt(STAT_ERROR**2 + sys_up**2)
TOTAL_DOWN = np.sqrt(STAT_ERROR**2 + sys_down**2)

print(f"Stat error:       ±{STAT_ERROR:.2f}%")
print(f"Sys up:           +{sys_up:.2f}%")
print(f"Sys down:         -{sys_down:.2f}%")
print(f"Total upper:      +{TOTAL_UP:.2f}%")
print(f"Total lower:      -{TOTAL_DOWN:.2f}%")


plot_official_xs(
    CV_val=2.04334e-38,
    CV_label='GENIE v3.0.6',
    data_val=3.23e-38,
    BNB_stat_err=STAT_ERROR,
    total_err_up=TOTAL_UP,
    total_err_down=TOTAL_DOWN,
    pot_string=r"$\mathbf{2.57\times10^{20}}$",
    label_prefix="Data",
    plot_name="plots/NCQE_XS.png",
    stat_only=False,
    deexc_alt_val=1.97893e-38,    # taken from GENIE flattree
    deexc_alt_label="Base GENIE de-exc."
)

Stat error:       ±3.60%
Sys up:           +38.39%
Sys down:         -43.29%
Total upper:      +38.56%
Total lower:      -43.44%
Plot saved to: plots/plots/NCQE_XS.png
